In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import missingno as msno
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import ttest_rel
from sklearn.compose import TransformedTargetRegressor

In [62]:
df_iso = pd.read_csv('iso.csv')

In [68]:
X = df_iso.drop(['Dev Time (Days)', 'issue_key','Current Status'], axis=1)
y = np.log1p(df_iso['Dev Time (Days)'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

r2 = r2_score(y_test, predictions)
print(f"R-squared Score: {r2:.2f}")

imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(imp.to_string(max_rows=None))

Model trained successfully!
Mean Absolute Error (MAE): 0.17
R-squared Score: 0.31
Start Date                                  5.417733e-01
total_swag                                  5.494973e-02
fix_version_ASTRO_SR2019.2                  2.344260e-02
impacted_area_Box Test                      2.084616e-02
impacted_area_CPS/RM                        1.701727e-02
technology_(none)                           1.680940e-02
impacted_area_Low Level                     1.650021e-02
fix_version_ASTRO_SR7.18.5                  1.380525e-02
subtask_count                               1.221841e-02
impacted_area_(none)                        1.168553e-02
One list                                    1.078621e-02
impacted_area_Ergo                          1.053047e-02
fix_version_(no version)                    1.035891e-02
technology_APX                              1.000548e-02
impacted_area_RC Insights                   9.046898e-03
impacted_area_SWBB Android Core / Common    8.941763e-03
impact

# Test Multiple Imputation Methods for total_swag (After Isolation Forest)

In [71]:
df = df_iso.copy()
num_cols = df.select_dtypes(include='number').columns
assert 'total_swag' in num_cols, "total_swag must be numeric."
df_num = df[num_cols]

In [72]:
obs_idx = df_num['total_swag'].dropna().index
rng = np.random.default_rng(42)
val_size = max(1, int(0.2 * len(obs_idx)))
val_idx = rng.choice(obs_idx, size=val_size, replace=False)
df_test = df_num.copy()
y_true = df_test.loc[val_idx, 'total_swag'].to_numpy()
df_test.loc[val_idx, 'total_swag'] = np.nan

Bayesian Ridge (features scaled)

In [73]:
est = make_pipeline(StandardScaler(), BayesianRidge())
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=est, sample_posterior = True, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 5662.843 RMSE: 352567702.466 Mean imputation SD: 6326.137


Linear Regression

In [74]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=LinearRegression(), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 7049.653 RMSE: 430895588.088 Mean imputation SD: 0.000


Decision Tree

In [75]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=DecisionTreeRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 3976.500 RMSE: 362571577.422 Mean imputation SD: 769.752


Random Forest

In [76]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=RandomForestRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 3826.614 RMSE: 339033151.089 Mean imputation SD: 226.707


Extra Trees

In [77]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=200, random_state=seed, n_jobs=-1), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 3862.961 RMSE: 341225285.972 Mean imputation SD: 140.068


Gradient boosting

In [78]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=GradientBoostingRegressor(random_state=seed), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 4106.616 RMSE: 348777783.443 Mean imputation SD: 70.574


K Neighbours

In [79]:
m = 20
imputed_preds =[]

for seed in range(m):
    imp = IterativeImputer(estimator=KNeighborsRegressor(n_neighbors=5), sample_posterior = False, random_state = seed, max_iter = 100, skip_complete=True)
    imputed = imp.fit_transform(df_test)
    imputed_df = pd.DataFrame(imputed, columns = df_test.columns, index = df_test.index)
    imputed_preds.append(imputed_df.loc[val_idx, 'total_swag'].to_numpy())

imputed_preds = np.vstack(imputed_preds)
y_pred = imputed_preds.mean(axis=0)
mae = mean_absolute_error(y_true,y_pred)
rmse = mean_squared_error(y_true, y_pred)
print(f"MAE: {mae:.3f} RMSE: {rmse:.3f} Mean imputation SD: {imputed_preds.std(axis=0).mean():.3f}")

MAE: 4907.653 RMSE: 347717540.453 Mean imputation SD: 0.000


Choose Random Forest

In [81]:
use_iso = df_iso.drop(['issue_key', 'Current Status'], axis=1)

imputer_iso = IterativeImputer(
    estimator = RandomForestRegressor(n_estimators=100, random_state=42),
    max_iter = 10,
    random_state = 42
)

df_iso_imputed_values = imputer_iso.fit_transform(use_iso)
df_iso_final = pd.DataFrame(df_iso_imputed_values, columns=use_iso.columns, index=use_iso.index)

In [83]:
df_iso_final.to_csv('imputed_iso.csv', index=False)